# 베이지안 최적화를 활용한 유도 파라미터 튜닝
# Bayesian Optimization for Guidance Parameter Tuning

---

> **개인 연구 노트** | Personal Research Note  
> 유도법칙의 파라미터를 자동으로 최적화할 수 있을까?  
> *Can we automatically tune guidance law parameters?*

---

## 연구 동기

APN(Augmented Proportional Navigation) 시뮬레이션을 돌리다 보면 항상 이런 의문이 생긴다:  
**"N=4가 최적인가? 아니면 교전 단계마다 다른 N을 쓰면 더 좋을까?"**

문제는 이걸 확인하려면 시뮬레이션을 여러 번 돌려야 한다는 것이다.  
6-DOF MC(Monte Carlo) 시뮬레이션 1회는 꽤 비싸다.  
그렇다면 *최소한의 시뮬레이션 횟수로 최적 파라미터를 찾는 방법은?*

이 노트북은 **Bayesian Optimization (BO)** 을 APN 파라미터 최적화에 적용해본 탐색적 연구이다.  
라이브러리 없이 numpy/scipy만으로 직접 구현하면서 원리를 이해하는 것이 목표다.

```
miss_distance = f(N_boost, N_mid, N_terminal)   ← expensive black-box function
목표: 최소 평가 횟수로 argmin 탐색
```

## 1. 왜 Bayesian Optimization인가?

### 문제 정의

파라미터 $\theta = (N_{\text{boost}}, N_{\text{mid}}, N_{\text{term}})$에 대해 목적함수:

$$f(\theta) = \mathbb{E}\left[\text{Miss Distance}(\theta)\right] \quad \text{over MC scenarios}$$

를 최소화하고 싶다. 이 함수의 특징:

| 특성 | 설명 |
|------|------|
| **Expensive** | 1회 평가 = MC 시뮬레이션 (≥ 수십 초) |
| **Black-box** | 해석적 gradient 없음 |
| **Noisy** | MC 샘플링으로 인한 랜덤 노이즈 포함 |
| **Low dimension** | 파라미터 수 3~10개 |

### 기존 방법의 한계

- **Grid Search**: 파라미터가 3개, 각 10포인트이면 $10^3 = 1000$번 평가 필요. 비현실적.
- **Random Search**: 운에 맡기는 탐색. 수렴 보장 없음.
- **Gradient-based**: Black-box 함수에 적용 불가. 시뮬레이션은 미분 불가.

### Bayesian Optimization의 아이디어

```
1. 지금까지 평가한 점들로 surrogate model (GP) 학습
2. GP의 불확실성 정보를 이용해 "어디를 다음에 평가할지" 결정 (acquisition function)
3. 그 점을 실제로 평가 → GP 업데이트 → 반복
```

> 시뮬레이션 1회가 비쌀 때 (6DOF MC) **최소 평가 횟수로 최적해 탐색**하는 것이 핵심.
> BO는 일반적으로 **20~50번** 안에 좋은 해를 찾는다.

## 2. Bayesian Optimization 이론

### 2.1 Surrogate Model: Gaussian Process (GP)

GP는 함수에 대한 **확률적 믿음(belief)** 을 모델링한다:

$$f(x) \sim \mathcal{GP}\left(m(x),\, k(x, x')\right)$$

- $m(x)$: 평균 함수 (보통 0으로 설정)
- $k(x, x')$: 커널 함수 — 두 점 간의 "유사도" 정의

**RBF (Radial Basis Function) 커널:**

$$k(x, x') = \sigma_f^2 \exp\left(-\frac{\|x - x'\|^2}{2\ell^2}\right)$$

관측 데이터 $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^n$가 주어졌을 때, 새 점 $x_*$에서의 예측:

$$\mu(x_*) = k_*^T (K + \sigma_n^2 I)^{-1} \mathbf{y}$$

$$\sigma^2(x_*) = k(x_*, x_*) - k_*^T (K + \sigma_n^2 I)^{-1} k_*$$

여기서 $k_* = [k(x_*, x_1), \ldots, k(x_*, x_n)]^T$, $K_{ij} = k(x_i, x_j)$.

### 2.2 Acquisition Functions

다음 평가 위치를 결정하는 함수. **Exploration vs Exploitation** 균형이 핵심.

**Expected Improvement (EI):**

$$\text{EI}(x) = \mathbb{E}\left[\max(f_{\text{best}} - f(x),\, 0)\right]$$

GP의 예측 분포가 $\mathcal{N}(\mu(x), \sigma^2(x))$일 때 (최소화 문제):

$$\text{EI}(x) = (f_{\text{best}} - \mu(x))\,\Phi(Z) + \sigma(x)\,\phi(Z)$$

$$Z = \frac{f_{\text{best}} - \mu(x)}{\sigma(x)}$$

여기서 $\Phi$는 표준정규 CDF, $\phi$는 PDF.

**Upper Confidence Bound (UCB):**

$$\text{UCB}(x) = -(\mu(x) - \kappa\,\sigma(x)) \quad (\text{minimize: negative UCB})$$

**Probability of Improvement (PI):**

$$\text{PI}(x) = \Phi\left(\frac{f_{\text{best}} - \mu(x)}{\sigma(x)}\right)$$

> EI가 가장 일반적으로 사용됨. Exploitation과 Exploration의 균형이 자연스럽게 달성됨.

In [ ]:
"""라이브러리 없이 직접 구현한 Bayesian Optimization
numpy, scipy, matplotlib만 사용"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (macOS)
plt.rcParams['font.family'] = ['AppleGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)


# ============================================================
# Gaussian Process Regressor (from scratch)
# ============================================================

class GaussianProcess:
    """RBF 커널 기반 Gaussian Process Regressor.
    
    Parameters
    ----------
    length_scale : float
        RBF 커널의 길이 스케일 (l)
    signal_var : float
        신호 분산 (sigma_f^2)
    noise_var : float
        관측 노이즈 분산 (sigma_n^2)
    """

    def __init__(self, length_scale=1.0, signal_var=1.0, noise_var=1e-6):
        self.l = length_scale
        self.sf2 = signal_var
        self.sn2 = noise_var
        self.X_train = None
        self.y_train = None
        self.L = None       # Cholesky factor
        self.alpha = None   # (K + sn2*I)^{-1} y

    def _rbf_kernel(self, X1, X2):
        """RBF (squared exponential) 커널 행렬 계산."""
        # X1: (n, d), X2: (m, d) → K: (n, m)
        X1 = np.atleast_2d(X1)
        X2 = np.atleast_2d(X2)
        diff = X1[:, np.newaxis, :] - X2[np.newaxis, :, :]  # (n, m, d)
        sq_dist = np.sum(diff**2, axis=-1)                   # (n, m)
        return self.sf2 * np.exp(-0.5 * sq_dist / self.l**2)

    def fit(self, X, y):
        """학습 데이터로 GP 업데이트 (Cholesky 분해 이용)."""
        self.X_train = np.atleast_2d(X)
        self.y_train = np.asarray(y).ravel()

        K = self._rbf_kernel(self.X_train, self.X_train)
        K_noisy = K + self.sn2 * np.eye(len(self.y_train))

        # Cholesky: K_noisy = L L^T
        try:
            self.L = np.linalg.cholesky(K_noisy)
        except np.linalg.LinAlgError:
            # 수치 안정성 보강
            K_noisy += 1e-6 * np.eye(len(self.y_train))
            self.L = np.linalg.cholesky(K_noisy)

        # alpha = (K + sn2*I)^{-1} y
        self.alpha = np.linalg.solve(self.L.T, np.linalg.solve(self.L, self.y_train))
        return self

    def predict(self, X_star, return_std=True):
        """새 점에서 GP 예측 (평균 + 표준편차)."""
        X_star = np.atleast_2d(X_star)

        k_star = self._rbf_kernel(X_star, self.X_train)  # (n*, n_train)
        mu = k_star @ self.alpha                          # (n*,)

        if not return_std:
            return mu

        # 분산: k(x*, x*) - k_*^T (K + sn2*I)^{-1} k_*
        v = np.linalg.solve(self.L, k_star.T)            # (n_train, n*)
        k_star_star = self._rbf_kernel(X_star, X_star)   # (n*, n*)
        var = np.diag(k_star_star) - np.sum(v**2, axis=0)
        var = np.maximum(var, 1e-10)   # 수치 오차로 음수 방지
        return mu, np.sqrt(var)


# ============================================================
# Acquisition Functions
# ============================================================

def expected_improvement(X, gp, f_best, xi=0.01):
    """Expected Improvement acquisition function (최소화 문제).
    
    EI(x) = (f_best - mu(x) - xi) * Phi(Z) + sigma(x) * phi(Z)
    """
    mu, sigma = gp.predict(X, return_std=True)

    # sigma가 0인 경우 처리
    with np.errstate(divide='ignore', invalid='ignore'):
        Z = (f_best - mu - xi) / sigma
        ei = (f_best - mu - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma < 1e-10] = 0.0

    return ei


def upper_confidence_bound(X, gp, kappa=2.0):
    """UCB acquisition function (최소화용 negative UCB)."""
    mu, sigma = gp.predict(X, return_std=True)
    return -(mu - kappa * sigma)   # 최소화 solver를 위해 음수


# ============================================================
# Bayesian Optimization Loop
# ============================================================

def maximize_acquisition(acq_func, bounds, n_restarts=10):
    """acquisition function 최대화 (다중 시작점 L-BFGS-B)."""
    best_x = None
    best_val = -np.inf
    dim = len(bounds)

    for _ in range(n_restarts):
        x0 = np.array([np.random.uniform(lo, hi) for lo, hi in bounds])
        result = minimize(
            lambda x: -acq_func(x.reshape(1, -1)).ravel()[0],
            x0,
            bounds=bounds,
            method='L-BFGS-B'
        )
        if -result.fun > best_val:
            best_val = -result.fun
            best_x = result.x

    return best_x


class BayesianOptimizer:
    """Bayesian Optimization 메인 클래스.
    
    Parameters
    ----------
    objective : callable
        목적함수 f(x) → scalar (최소화 목표)
    bounds : list of (lo, hi) tuples
        파라미터 탐색 범위
    n_init : int
        초기 랜덤 샘플 수
    """

    def __init__(self, objective, bounds, n_init=5,
                 length_scale=1.0, signal_var=1.0, noise_var=0.01):
        self.objective = objective
        self.bounds = bounds
        self.dim = len(bounds)
        self.n_init = n_init
        self.gp = GaussianProcess(
            length_scale=length_scale,
            signal_var=signal_var,
            noise_var=noise_var
        )
        self.X_obs = []
        self.y_obs = []
        self.history = []   # (iteration, x, y, best_so_far)

    def _random_sample(self):
        return np.array([np.random.uniform(lo, hi) for lo, hi in self.bounds])

    def initialize(self):
        """초기 랜덤 평가."""
        print(f"초기화: {self.n_init}개 랜덤 샘플 평가 중...")
        for i in range(self.n_init):
            x = self._random_sample()
            y = self.objective(x)
            self.X_obs.append(x)
            self.y_obs.append(y)
            best = min(self.y_obs)
            self.history.append((i, x.copy(), y, best))
        self.gp.fit(np.array(self.X_obs), np.array(self.y_obs))
        print(f"  초기 최솟값: {min(self.y_obs):.4f}")

    def step(self, iteration, acq_type='EI'):
        """1 iteration: acquisition 최대화 → 평가 → GP 업데이트."""
        f_best = min(self.y_obs)

        if acq_type == 'EI':
            acq_func = lambda X: expected_improvement(X, self.gp, f_best)
        else:
            acq_func = lambda X: upper_confidence_bound(X, self.gp)

        x_next = maximize_acquisition(acq_func, self.bounds, n_restarts=15)
        y_next = self.objective(x_next)

        self.X_obs.append(x_next)
        self.y_obs.append(y_next)
        self.gp.fit(np.array(self.X_obs), np.array(self.y_obs))

        best = min(self.y_obs)
        self.history.append((iteration, x_next.copy(), y_next, best))
        return x_next, y_next, best

    def run(self, n_iter=20, acq_type='EI', verbose=True):
        """전체 BO 루프 실행."""
        self.initialize()
        print(f"\nBO 시작: {n_iter} iterations, acquisition={acq_type}")
        print("-" * 50)
        for i in range(n_iter):
            x_next, y_next, best = self.step(i + self.n_init, acq_type)
            if verbose and (i % 5 == 0 or i == n_iter - 1):
                print(f"  iter {i+1:3d}/{n_iter} | f(x)={y_next:.4f} | best={best:.4f}")
        print("-" * 50)
        best_idx = np.argmin(self.y_obs)
        print(f"최적 파라미터: {self.X_obs[best_idx]}")
        print(f"최솟값: {self.y_obs[best_idx]:.4f}")
        return self.X_obs[best_idx], self.y_obs[best_idx]

    @property
    def best_x(self):
        idx = np.argmin(self.y_obs)
        return np.array(self.X_obs[idx])

    @property
    def best_y(self):
        return min(self.y_obs)


print("GP + EI 구현 완료")
print(f"  GaussianProcess: RBF 커널, Cholesky 분해")
print(f"  Acquisition: EI, UCB")
print(f"  Optimizer: L-BFGS-B with {15} restarts")

In [ ]:
"""1D 검증 실험: 알려진 다봉 함수(multimodal)에서 BO 동작 확인

테스트 함수: f(x) = sin(3x) + x^2 - 0.7*x  (여러 극솟값 포함)
"""

# ── 1D 테스트 함수 정의 ──────────────────────────────────────────
def test_func_1d(x):
    x = np.asarray(x).ravel()[0]
    return np.sin(3 * x) + x**2 - 0.7 * x


# 참값 그리드
x_grid = np.linspace(-2, 4, 500)
y_true = np.array([test_func_1d(xi) for xi in x_grid])
x_opt_true = x_grid[np.argmin(y_true)]
y_opt_true = y_true.min()

# ── BO 실행 (1D) ─────────────────────────────────────────────────
np.random.seed(7)
bo_1d = BayesianOptimizer(
    objective=test_func_1d,
    bounds=[(-2, 4)],
    n_init=3,
    length_scale=0.8,
    signal_var=1.5,
    noise_var=1e-4
)
bo_1d.run(n_iter=17, acq_type='EI', verbose=False)

# ── 스텝별 시각화 ─────────────────────────────────────────────────
plot_iters = [3, 7, 12, 20]   # 초기화 후 총 관측 수 (init=3 포함)
fig, axes = plt.subplots(len(plot_iters), 1, figsize=(10, 14))
fig.suptitle('Bayesian Optimization 진행 과정 (1D 검증)', fontsize=14, fontweight='bold', y=1.01)

# 각 스텝에서 GP surrogate + acquisition 재구성
snap_bo = BayesianOptimizer(
    objective=test_func_1d,
    bounds=[(-2, 4)],
    n_init=3,
    length_scale=0.8,
    signal_var=1.5,
    noise_var=1e-4
)
np.random.seed(7)
snap_bo.initialize()

snap_states = []
snap_states.append((list(snap_bo.X_obs), list(snap_bo.y_obs)))

for i in range(17):
    snap_bo.step(i + snap_bo.n_init, 'EI')
    snap_states.append((list(snap_bo.X_obs), list(snap_bo.y_obs)))

# 그리기
x_plot = x_grid.reshape(-1, 1)

for ax_idx, n_obs in enumerate(plot_iters):
    ax = axes[ax_idx]

    state_idx = min(n_obs - 3, len(snap_states) - 1)
    X_s, y_s = snap_states[state_idx]

    # GP 재학습
    gp_snap = GaussianProcess(length_scale=0.8, signal_var=1.5, noise_var=1e-4)
    gp_snap.fit(np.array(X_s), np.array(y_s))

    mu_s, std_s = gp_snap.predict(x_plot)
    f_best_s = min(y_s)
    ei_s = expected_improvement(x_plot, gp_snap, f_best_s)

    # 참 함수
    ax.plot(x_grid, y_true, 'k--', lw=1.5, alpha=0.5, label='True f(x)')
    ax.axvline(x_opt_true, color='gray', lw=1, ls=':', alpha=0.6)

    # GP 예측
    ax.plot(x_grid, mu_s, 'b-', lw=2, label='GP 평균')
    ax.fill_between(x_grid, mu_s - 2*std_s, mu_s + 2*std_s,
                    alpha=0.15, color='blue', label='95% CI')

    # 관측 포인트
    X_arr = np.array(X_s).ravel()
    y_arr = np.array(y_s)
    ax.scatter(X_arr, y_arr, c='red', zorder=5, s=50, label='관측점')
    ax.scatter(X_arr[-1], y_arr[-1], c='orange', zorder=6, s=100,
               marker='*', label='최신 선택')

    # Acquisition (스케일 조정하여 같이 표시)
    ax2 = ax.twinx()
    ax2.fill_between(x_grid, 0, ei_s.ravel(), alpha=0.2, color='green')
    ax2.plot(x_grid, ei_s.ravel(), 'g-', lw=1, alpha=0.7, label='EI')
    ax2.set_ylabel('EI', color='green', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='green')

    ax.set_title(f'Iteration {n_obs - 3} | 총 관측 수: {len(X_s)} | Best: {f_best_s:.4f}',
                 fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('f(x)')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xlim(-2, 4)

plt.tight_layout()
plt.savefig('bo_1d_validation.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n참 최솟값: x={x_opt_true:.4f}, f={y_opt_true:.4f}")
print(f"BO 탐색 결과: x={bo_1d.best_x[0]:.4f}, f={bo_1d.best_y:.4f}")
print(f"오차: {abs(bo_1d.best_x[0] - x_opt_true):.4f}")

## 3. 유도 파라미터 최적화 문제

### 3.1 APN과 Navigation Constant N

Augmented Proportional Navigation (APN) 가속도 명령:

$$a_c = N \cdot V_c \cdot \dot{\lambda} + \frac{N}{2} a_T$$

여기서 $N$은 **Navigation Constant** (보통 3~6). 이 값이 성능에 직접적인 영향을 미친다.

### 3.2 N Gain Scheduling 아이디어

교전 단계를 3구간으로 나누고, 구간별로 다른 N을 적용하면 어떨까?

```
Boost 단계   (초기 가속): N_boost  — 빠른 초기 조준
Midcourse 단계 (순항):   N_mid    — 에너지 절약
Terminal 단계  (종말):   N_term   — 정밀 유도
```

**최적화 문제:**

$$\min_{N_{\text{boost}}, N_{\text{mid}}, N_{\text{term}}} \quad \mathbb{E}_{\text{MC}}\left[\text{miss}(N_{\text{boost}}, N_{\text{mid}}, N_{\text{term}})\right]$$

$$\text{s.t.} \quad N_i \in [2, 8], \quad i \in \{\text{boost, mid, term}\}$$

### 3.3 시뮬레이션 모델 선택

실제 6DOF 시뮬레이션은 너무 느리므로, 2D 점질량 모델(point-mass)로 간략화:
- 2D 평면 교전 (elevation 고정)
- 1차 지연 자동조종 (autopilot lag)
- MC: 표적 기동 불확실성 + 초기 조건 분산

> "N을 시간에 따라 변화시키면 (gain scheduling) 더 좋은 결과가 나올 수 있다"는 가설을 검증해보자.

In [ ]:
"""2D 간략화 교전 시뮬레이터 + N gain scheduling MC 평가함수"""

# ── 상수 ─────────────────────────────────────────────────────────
DT     = 0.01    # s
T_F    = 10.0    # s (최대 비행시간)
VM     = 300.0   # m/s 유도탄 속도
VT     = 100.0   # m/s 표적 속도 (기준)
TAU_AP = 0.3     # s  자동조종 시상수 (1차 지연)
A_LIM  = 30.0 * 9.81   # m/s^2 가속도 포화 한계 (30g)
N_MC   = 20      # MC 횟수


def apn_engagement_2d(N_sched, seed=None, verbose=False):
    """2D APN 교전 시뮬레이션. N_sched = (N_boost, N_mid, N_term).
    
    Returns
    -------
    miss : float  [m]
    traj : dict   (선택적, verbose=True 시)
    """
    rng = np.random.default_rng(seed)

    N_boost, N_mid, N_term = N_sched

    # --- 초기 조건 (MC 분산 포함) ---
    R0 = 8000.0 + rng.normal(0, 200)     # m  초기 사거리
    # 미사일: (0, 0) 향해 +x 방향 발사
    Mx, My = 0.0, 0.0
    Vmy = rng.normal(0, VM * 0.02)        # 발사 각도 오차
    Vmx = np.sqrt(max(VM**2 - Vmy**2, 1.0))

    # 표적: x=R0, y=0 근처에서 횡방향 기동
    Tx = R0 + rng.normal(0, 100)
    Ty = rng.normal(0, 300)              # 초기 횡위치 분산
    Vt_heading = np.pi + rng.normal(0, np.deg2rad(20))
    Vtx = VT * np.cos(Vt_heading)
    Vty = VT * np.sin(Vt_heading)
    aT_mag = rng.uniform(0, 5.0 * 9.81)   # 표적 기동 크기 [0, 5g]
    aT_dir = rng.choice([-1, 1])

    # 가속도 상태 (1차 지연 자동조종)
    a_ach = 0.0
    a_cmd = 0.0

    # 비행 시간 분할 (N gain scheduling)
    t_boost_end    = 0.2 * T_F   # 20% boost
    t_mid_end      = 0.7 * T_F   # 70% midcourse
    # 나머지 30% terminal

    traj_t, traj_miss = [], []

    t = 0.0
    while t < T_F:
        # 상대 벡터
        rx = Tx - Mx
        ry = Ty - My
        R  = np.sqrt(rx**2 + ry**2)

        if R < 5.0:   # 근접 폭발 조건
            break

        # LOS 각 및 LOS 변화율
        lam = np.arctan2(ry, rx)
        Vrx = Vtx - Vmx
        Vry = Vty - Vmy
        Vc  = -(rx * Vrx + ry * Vry) / R   # 닫힘 속도 (양수 = 접근)
        lam_dot = (rx * Vry - ry * Vrx) / R**2

        # N gain scheduling
        if t < t_boost_end:
            N = N_boost
        elif t < t_mid_end:
            N = N_mid
        else:
            N = N_term

        # APN 가속도 명령 (표적 기동 aT 포함)
        aT_perp = aT_mag * aT_dir   # 표적 횡기동
        a_cmd_new = N * Vc * lam_dot + 0.5 * N * aT_perp
        a_cmd_new = np.clip(a_cmd_new, -A_LIM, A_LIM)

        # 1차 지연 자동조종
        a_ach += DT * (a_cmd_new - a_ach) / TAU_AP

        # 상태 적분
        # 가속도는 LOS 수직 방향 (lam + 90deg)
        perp_x = -np.sin(lam)
        perp_y =  np.cos(lam)

        Vmx += DT * a_ach * perp_x
        Vmy += DT * a_ach * perp_y
        spd = np.sqrt(Vmx**2 + Vmy**2)
        if spd > 0:
            Vmx *= VM / spd   # 속도 크기 고정 (thrust)
            Vmy *= VM / spd

        Vtx += DT * 0    # 표적은 등속 (단순화)
        Vty += DT * 0

        Mx += DT * Vmx
        My += DT * Vmy
        Tx += DT * Vtx
        Ty += DT * Vty

        t += DT
        if verbose:
            traj_t.append(t)
            traj_miss.append(np.sqrt((Tx-Mx)**2 + (Ty-My)**2))

    miss = np.sqrt((Tx - Mx)**2 + (Ty - My)**2)

    if verbose:
        return miss, {'t': traj_t, 'range': traj_miss}
    return miss


def mean_miss_distance(N_params):
    """BO 목적함수: N_params = [N_boost, N_mid, N_term].
    N_MC번 MC 시뮬레이션의 평균 miss distance 반환.
    """
    N_boost, N_mid, N_term = N_params[0], N_params[1], N_params[2]
    misses = []
    for i in range(N_MC):
        m = apn_engagement_2d((N_boost, N_mid, N_term), seed=i)
        misses.append(m)
    return float(np.mean(misses))


# ── 빠른 기능 검증 ────────────────────────────────────────────────
miss_test, traj_test = apn_engagement_2d((4.0, 4.0, 4.0), seed=0, verbose=True)
print(f"단일 교전 miss distance (N=4 고정): {miss_test:.2f} m")
print(f"비행 시간: {len(traj_test['t']) * DT:.2f} s")

# 고정 N=4 MC 기준선
miss_baseline = mean_miss_distance([4.0, 4.0, 4.0])
print(f"\n기준선 (N=4 고정): 평균 miss = {miss_baseline:.2f} m  ({N_MC}회 MC)")

In [ ]:
"""BO 실행: N gain scheduling 최적화 (30 iterations)"""

np.random.seed(0)

# N 범위: 2 ~ 8 (각 단계 독립)
bounds_N = [(2.0, 8.0), (2.0, 8.0), (2.0, 8.0)]

bo_guidance = BayesianOptimizer(
    objective=mean_miss_distance,
    bounds=bounds_N,
    n_init=5,
    length_scale=1.5,
    signal_var=50.0,      # miss distance의 스케일에 맞춤
    noise_var=1.0         # MC 노이즈 반영
)

best_params, best_miss = bo_guidance.run(n_iter=25, acq_type='EI', verbose=True)

print(f"\n=" * 40)
print(f"최적 N schedule:")
print(f"  N_boost    = {best_params[0]:.3f}")
print(f"  N_midcourse= {best_params[1]:.3f}")
print(f"  N_terminal = {best_params[2]:.3f}")
print(f"  최적 miss distance: {best_miss:.2f} m")
print(f"  기준 (N=4 고정):    {miss_baseline:.2f} m")
improvement = (miss_baseline - best_miss) / miss_baseline * 100
print(f"  개선율: {improvement:.1f}%")

# ── 수렴 그래프 ───────────────────────────────────────────────────
iters   = [h[0] for h in bo_guidance.history]
y_vals  = [h[2] for h in bo_guidance.history]
best_so = [h[3] for h in bo_guidance.history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BO 최적화 수렴 과정', fontsize=13, fontweight='bold')

# 왼쪽: 수렴 곡선
ax = axes[0]
ax.scatter(range(len(y_vals)), y_vals, alpha=0.5, s=30, color='steelblue', label='f(x) 각 평가')
ax.plot(range(len(best_so)), best_so, 'r-', lw=2.5, label='최솟값 (best so far)')
ax.axhline(miss_baseline, color='gray', ls='--', lw=1.5, label=f'기준 N=4 ({miss_baseline:.1f}m)')
ax.axhline(best_miss, color='orange', ls=':', lw=1.5, label=f'BO 최적 ({best_miss:.1f}m)')
ax.axvspan(bo_guidance.n_init, bo_guidance.n_init, color='k', alpha=0.1)
ax.axvline(bo_guidance.n_init - 0.5, color='black', ls=':', lw=1, alpha=0.5)
ax.text(1, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 1,
        '초기화', fontsize=9, color='gray')
ax.set_xlabel('Iteration')
ax.set_ylabel('평균 Miss Distance [m]')
ax.set_title('수렴 곡선')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 오른쪽: N schedule 파라미터 추적
ax2 = axes[1]
X_obs_arr = np.array(bo_guidance.X_obs)
colors = ['#2196F3', '#4CAF50', '#FF5722']
labels = ['$N_{\\mathrm{boost}}$', '$N_{\\mathrm{mid}}$', '$N_{\\mathrm{term}}$']

for j in range(3):
    ax2.scatter(range(len(X_obs_arr)), X_obs_arr[:, j], alpha=0.5, s=25,
                color=colors[j], label=labels[j])

# 최적 파라미터 표시
for j in range(3):
    ax2.axhline(best_params[j], color=colors[j], ls='--', lw=1.5, alpha=0.7)

ax2.set_xlabel('Iteration')
ax2.set_ylabel('N 값')
ax2.set_title('파라미터 샘플링 이력')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_ylim(1.5, 8.5)

plt.tight_layout()
plt.savefig('bo_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

# 수렴 체크
late_iters = best_so[20:]  # 20번째 이후
if len(late_iters) > 1:
    late_var = np.std(late_iters)
    print(f"\n20번째 iteration 이후 best 표준편차: {late_var:.4f} m")
    if late_var < 0.5:
        print("  → 수렴 확인됨 (변동 < 0.5m)")

## 4. 최적화 결과 분석

### 관찰 사항

BO 결과에서 주목할 점:

1. **Terminal N이 가장 중요**: $N_{\text{term}}$이 높을수록 종말 단계 정밀도 향상  
   → 종말 단계에서 LOS rate가 크게 변하기 때문에 높은 gain이 유리

2. **Midcourse N은 낮아도 됨**: 에너지 효율성과 accel 포화를 피하기 위해  
   → 교전 중간에는 큰 명령 불필요

3. **Boost N은 중간값**: 초기 조준이 너무 aggressiv하면 accel 포화 발생

4. **수렴 속도**: 약 15-20번 iteration 내에 안정적인 최솟값 도달  
   → Grid search 대비 훨씬 효율적

### 개선율 해석

$$\text{개선율} = \frac{\text{miss}_{N=4} - \text{miss}_{\text{opt}}}{\text{miss}_{N=4}} \times 100\%$$

gain scheduling 자체의 자유도 증가 덕분에 성능 향상이 가능했다.  
실전에서는 표적 기동 패턴, 공력 제약 등 더 많은 요소가 있지만, 방향성은 확인됐다.

In [ ]:
"""결과 시각화: 최적 N schedule vs 고정 N=4 비교"""

N_COMPARE_MC = 100   # 최종 비교는 더 많은 MC 샘플

# 고정 N=4
miss_fixed = [apn_engagement_2d((4.0, 4.0, 4.0), seed=i) for i in range(N_COMPARE_MC)]

# 최적 N schedule
miss_opt   = [apn_engagement_2d(tuple(best_params), seed=i) for i in range(N_COMPARE_MC)]

miss_fixed = np.array(miss_fixed)
miss_opt   = np.array(miss_opt)

mean_fixed = miss_fixed.mean()
mean_opt   = miss_opt.mean()
improvement_final = (mean_fixed - mean_opt) / mean_fixed * 100

print(f"최종 비교 ({N_COMPARE_MC}회 MC):")
print(f"  고정 N=4 평균 miss:   {mean_fixed:.2f} m")
print(f"  최적 schedule 평균:   {mean_opt:.2f} m")
print(f"  개선율:              {improvement_final:.1f}%")

# ── 그래프 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('최적화 결과 비교: 고정 N=4 vs 최적 N schedule', fontsize=13, fontweight='bold')

# 1) Miss distance 히스토그램
ax = axes[0]
bins = np.linspace(0, max(miss_fixed.max(), miss_opt.max()) * 1.1, 25)
ax.hist(miss_fixed, bins=bins, alpha=0.6, color='steelblue', label=f'N=4 고정 (μ={mean_fixed:.1f}m)', edgecolor='white')
ax.hist(miss_opt,   bins=bins, alpha=0.6, color='tomato',    label=f'최적 schedule (μ={mean_opt:.1f}m)', edgecolor='white')
ax.axvline(mean_fixed, color='steelblue', ls='--', lw=2)
ax.axvline(mean_opt,   color='tomato',    ls='--', lw=2)
ax.set_xlabel('Miss Distance [m]')
ax.set_ylabel('빈도')
ax.set_title('Miss Distance 분포')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# 2) 최적 N schedule 시각화
ax2 = axes[1]
t_total = 10.0
t_segments = [0, 0.2*t_total, 0.7*t_total, t_total]
N_vals = [best_params[0], best_params[1], best_params[2]]
colors_seg = ['#FF7043', '#66BB6A', '#42A5F5']
labels_seg = ['Boost', 'Midcourse', 'Terminal']

for i in range(3):
    ax2.fill_between([t_segments[i], t_segments[i+1]], 0, N_vals[i],
                     alpha=0.4, color=colors_seg[i])
    ax2.plot([t_segments[i], t_segments[i+1]], [N_vals[i], N_vals[i]],
             color=colors_seg[i], lw=2.5, label=f'{labels_seg[i]}: N={N_vals[i]:.2f}')

ax2.axhline(4.0, color='gray', ls='--', lw=1.5, alpha=0.7, label='기준 N=4')
for t in t_segments[1:-1]:
    ax2.axvline(t, color='black', ls=':', lw=1, alpha=0.5)

ax2.set_xlabel('비행 시간 [s]')
ax2.set_ylabel('Navigation Constant N')
ax2.set_title('최적 N Gain Schedule')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 9)
ax2.set_xlim(0, t_total)

# 3) 누적 분포 비교 (CDF)
ax3 = axes[2]
x_sorted_f = np.sort(miss_fixed)
x_sorted_o = np.sort(miss_opt)
cdf_y = np.linspace(0, 1, len(miss_fixed))

ax3.plot(x_sorted_f, cdf_y, 'b-', lw=2.5, label='N=4 고정')
ax3.plot(x_sorted_o, cdf_y, 'r-', lw=2.5, label='최적 schedule')
ax3.axhline(0.5, color='gray', ls=':', lw=1, alpha=0.7, label='중앙값 기준선')
ax3.set_xlabel('Miss Distance [m]')
ax3.set_ylabel('누적 확률')
ax3.set_title('누적 분포 함수 (CDF)')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# 개선율 텍스트 추가
ax3.text(0.98, 0.05, f'개선율: {improvement_final:.1f}%',
         transform=ax3.transAxes, ha='right', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('bo_result_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# 통계 요약
print(f"\n통계 요약:")
print(f"{'':20s} {'N=4 고정':>15s} {'최적 schedule':>15s}")
print("-" * 52)
stats = [
    ('평균 [m]',    miss_fixed.mean(),                miss_opt.mean()),
    ('표준편차 [m]', miss_fixed.std(),                 miss_opt.std()),
    ('중앙값 [m]',  np.median(miss_fixed),             np.median(miss_opt)),
    ('90th pct [m]', np.percentile(miss_fixed, 90),   np.percentile(miss_opt, 90)),
]
for name, v_fixed, v_opt in stats:
    print(f"{name:20s} {v_fixed:>15.2f} {v_opt:>15.2f}")

## 5. Multi-objective 확장 가능성

### 실제 설계에서의 고민

단순히 miss distance만 최소화하면 되는 게 아니다.  
높은 N은 miss를 줄이지만, **가속도 사용량(peak acceleration)** 을 늘린다.

- 가속도 포화 → 실제로는 명령 추종 불가  
- 과도한 기동 → 구조하중 증가  
- 에너지 소모 → 사거리 감소

### Pareto Front

두 목적이 상충(conflict)할 때, 하나도 희생하지 않고 개선할 수 없는 점들의 집합:

$$\mathcal{P}^* = \left\{ \theta : \nexists\, \theta'\, \text{s.t.}\; f_1(\theta') \leq f_1(\theta) \;\wedge\; f_2(\theta') \leq f_2(\theta) \right\}$$

설계자는 Pareto front 위에서 **가중치(trade-off preference)** 에 따라 운용점 선택.

### 확장 방향

- **NSGA-II, MOBO**: Multi-objective BO (예: EHVI — Expected Hypervolume Improvement)
- **Scalarization**: $\min \alpha \cdot \text{miss} + (1-\alpha) \cdot \text{accel}$ 로 단순화
- **Constraint handling**: 가속도 $\leq$ 30g를 hard constraint로 처리

> 실제 설계에서는 다목적 최적화가 필요하다. miss/accel/maneuverability의 균형이 핵심.

In [ ]:
"""Multi-objective 시각화: miss distance vs peak acceleration"""

def engagement_with_accel(N_sched, seed=0):
    """miss distance + peak acceleration 동시 반환."""
    N_boost, N_mid, N_term = N_sched
    rng = np.random.default_rng(seed)

    R0  = 8000.0
    Mx, My = 0.0, 0.0
    Vmx, Vmy = VM, 0.0
    Tx, Ty = R0, 0.0
    Vtx = -VT
    Vty = 0.0
    aT_mag = 3.0 * 9.81
    aT_dir = 1

    a_ach = 0.0
    t_boost_end = 0.2 * T_F
    t_mid_end   = 0.7 * T_F

    peak_a = 0.0
    t = 0.0

    while t < T_F:
        rx, ry = Tx - Mx, Ty - My
        R = np.sqrt(rx**2 + ry**2)
        if R < 5.0:
            break

        lam     = np.arctan2(ry, rx)
        Vrx, Vry = Vtx - Vmx, Vty - Vmy
        Vc       = -(rx*Vrx + ry*Vry) / R
        lam_dot  = (rx*Vry - ry*Vrx) / R**2

        N = N_boost if t < t_boost_end else (N_mid if t < t_mid_end else N_term)

        a_cmd = np.clip(N*Vc*lam_dot + 0.5*N*aT_mag*aT_dir, -A_LIM, A_LIM)
        a_ach += DT * (a_cmd - a_ach) / TAU_AP
        peak_a = max(peak_a, abs(a_ach))

        perp_x, perp_y = -np.sin(lam), np.cos(lam)
        Vmx += DT * a_ach * perp_x
        Vmy += DT * a_ach * perp_y
        spd = np.sqrt(Vmx**2 + Vmy**2)
        if spd > 0:
            Vmx *= VM/spd; Vmy *= VM/spd

        Mx += DT*Vmx; My += DT*Vmy
        Tx += DT*Vtx; Ty += DT*Vty
        t  += DT

    miss   = np.sqrt((Tx-Mx)**2 + (Ty-My)**2)
    return miss, peak_a / 9.81   # [m], [g]


# ── N 값 스윕하여 두 목적함수 평가 ───────────────────────────────
N_sweep = np.linspace(2, 8, 25)
results_mo = []   # (miss, peak_g)

# 고정 N (N_boost=N_mid=N_term=N)
for N in N_sweep:
    m, a = engagement_with_accel((N, N, N), seed=0)
    results_mo.append((m, a, N, 'fixed'))

# N gain scheduling 변형들 (N_term 변화, 나머지 고정)
scheduling_variants = []
for Nt in np.linspace(3, 8, 20):
    m, a = engagement_with_accel((3.0, 3.5, Nt), seed=0)
    scheduling_variants.append((m, a, Nt, 'sched'))

results_arr  = np.array([(r[0], r[1]) for r in results_mo])
sched_arr    = np.array([(r[0], r[1]) for r in scheduling_variants])

# ── Pareto front 추출 ─────────────────────────────────────────────
def is_pareto_efficient(costs):
    """Pareto-efficient 인덱스 반환 (최소화)."""
    n = len(costs)
    is_eff = np.ones(n, dtype=bool)
    for i in range(n):
        if is_eff[i]:
            # i를 지배하는 다른 점이 있으면 제거
            dominated = np.all(costs <= costs[i], axis=1) & np.any(costs < costs[i], axis=1)
            is_eff[dominated] = False
            is_eff[i] = True   # 자기 자신은 유지
    return is_eff

all_points = np.vstack([results_arr, sched_arr])
pareto_mask = is_pareto_efficient(all_points)
pareto_pts  = all_points[pareto_mask]
pareto_pts  = pareto_pts[pareto_pts[:, 0].argsort()]   # miss 기준 정렬

# ── 시각화 ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Multi-objective: Miss Distance vs Peak Acceleration', fontsize=13, fontweight='bold')

# 왼쪽: 산점도 + Pareto front
ax = axes[0]
sc1 = ax.scatter(results_arr[:, 0], results_arr[:, 1],
                 c=[r[2] for r in results_mo], cmap='Blues',
                 s=60, alpha=0.8, label='고정 N (N=2~8)', edgecolors='navy', lw=0.5)
sc2 = ax.scatter(sched_arr[:, 0], sched_arr[:, 1],
                 c=[r[2] for r in scheduling_variants], cmap='Reds',
                 s=60, alpha=0.8, marker='s', label='N schedule (N_term 변화)', edgecolors='darkred', lw=0.5)
ax.plot(pareto_pts[:, 0], pareto_pts[:, 1], 'k-o', lw=2, ms=6, label='Pareto Front', zorder=5)

# 최적 BO 결과 표시
m_opt_pt, a_opt_pt = engagement_with_accel(tuple(best_params), seed=0)
ax.scatter(m_opt_pt, a_opt_pt/9.81, c='gold', s=200, marker='*',
           zorder=6, label=f'BO 최적 (N={best_params[0]:.1f},{best_params[1]:.1f},{best_params[2]:.1f})',
           edgecolors='darkorange', lw=1.5)

plt.colorbar(sc1, ax=ax, label='N 값 (고정)')
ax.set_xlabel('Miss Distance [m]')
ax.set_ylabel('Peak Acceleration [g]')
ax.set_title('목적함수 공간 (Objective Space)')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.3)
ax.annotate('이 영역이 Pareto-optimal\n(두 목적 동시 개선 불가)',
            xy=(pareto_pts[len(pareto_pts)//2, 0], pareto_pts[len(pareto_pts)//2, 1]),
            xytext=(pareto_pts[:, 0].max() * 0.6, pareto_pts[:, 1].max() * 0.6),
            fontsize=8, color='black',
            arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

# 오른쪽: N에 따른 두 목적함수 추이
ax2 = axes[1]
ax2_r = ax2.twinx()

ax2.plot(N_sweep, results_arr[:, 0], 'b-o', lw=2, ms=5, label='Miss Distance')
ax2_r.plot(N_sweep, results_arr[:, 1], 'r-s', lw=2, ms=5, label='Peak Accel [g]')

ax2.axvline(4.0, color='gray', ls='--', lw=1.5, alpha=0.7, label='N=4 기준')
ax2.set_xlabel('Navigation Constant N')
ax2.set_ylabel('Miss Distance [m]', color='blue')
ax2_r.set_ylabel('Peak Acceleration [g]', color='red')
ax2.tick_params(axis='y', labelcolor='blue')
ax2_r.tick_params(axis='y', labelcolor='red')
ax2.set_title('N에 따른 성능 trade-off')

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_r.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bo_multiobjective.png', dpi=100, bbox_inches='tight')
plt.show()

print("Pareto front 포인트 수:", len(pareto_pts))
print("miss가 낮은 Pareto 포인트 (상위 3개):")
for pt in pareto_pts[:3]:
    print(f"  miss={pt[0]:.2f}m, peak_accel={pt[1]:.2f}g")

## 6. 한계와 향후 과제

### 현재 접근법의 한계

#### 차원의 저주 (Curse of Dimensionality)

| 파라미터 수 | BO 적용 평가 | 비고 |
|------------|------------|------|
| 3개 (본 연구) | 30~50회 | 실용적 |
| 5~10개 | 100~200회 | 가능하지만 느림 |
| 10개 이상 | 기하급수적 증가 | GP 자체의 한계 |

고차원에서는 GP 커널 행렬의 역행렬 계산이 $\mathcal{O}(n^3)$ 으로 병목이 된다.

#### Noisy Objective

MC 샘플 수가 적으면 ($N_{MC} < 50$) 노이즈가 커서 GP가 참 함수를 잘못 모델링할 수 있다.  
→ `noise_var` 하이퍼파라미터 튜닝이 중요하다.

#### 커널 설계

RBF 커널은 "부드러운" 함수를 가정한다. 유도 시뮬레이션처럼 불연속(포화, 충돌 판정)이 있으면  
Matern 커널이 더 적합할 수 있다.

### 향후 연구 방향

1. **Multi-fidelity BO**: 저충실도(2D 점질량) + 고충실도(6DOF) 혼합 사용  
   → 비싼 고충실도 평가를 최소화하면서 정확도 유지

2. **MOBO (Multi-objective BO)**: EHVI (Expected Hypervolume Improvement)  
   → miss/accel/maneuverability Pareto front 자동 탐색

3. **Transfer Learning BO**: 유사한 표적 유형에서 배운 GP를 새 교전에 전이  
   → 초기 수렴 속도 가속

4. **Online BO**: 비행 중 실시간으로 파라미터 업데이트  
   → 단, 계산 비용이 제약

> "실전 적용 시 시뮬레이션 비용을 줄이는 것이 핵심"  
> → Multi-fidelity BO (저충실도 모델 활용)도 연구 가치가 있다.

## 7. 정리

---

### 핵심 요약

| 항목 | 내용 |
|------|------|
| **방법** | Gaussian Process 기반 Bayesian Optimization |
| **적용** | APN Navigation Constant N gain scheduling |
| **파라미터** | $N_{\text{boost}}, N_{\text{mid}}, N_{\text{term}} \in [2, 8]$ |
| **평가 횟수** | 30회 (Grid search 1000회 대비) |
| **구현** | numpy/scipy만으로 from scratch |

### 배운 점

1. **BO는 시뮬레이션 기반 파라미터 튜닝에 효과적**  
   Black-box, noisy, expensive 세 조건을 모두 만족하는 유도 시뮬레이션에 잘 맞는다.

2. **유도 설계에서 자동화된 최적화 도구로서의 가능성**  
   N=4라는 경험적 값 대신, BO로 교전 조건에 맞게 자동 튜닝하는 흐름이 가능하다.

3. **GP 직접 구현의 의미**  
   Cholesky 분해, EI acquisition, L-BFGS-B 최적화를 직접 구현하면서  
   BO가 "왜" 잘 동작하는지 이해할 수 있었다.

4. **Multi-objective 문제는 더 복잡**  
   miss만 최소화하는 것으로는 부족하다. 가속도 제약과의 trade-off가 반드시 고려되어야 한다.

---

> *"좋은 유도법칙은 설계자의 직관 + 체계적인 최적화 도구의 조합에서 나온다."*  
> BO는 그 도구 중 하나가 될 수 있다.

---

**참고 자료**
- Brochu, E., Cora, V. M., & de Freitas, N. (2010). *A tutorial on Bayesian optimization of expensive cost functions.* arXiv:1012.2599
- Rasmussen, C. E. & Williams, C. K. I. (2006). *Gaussian Processes for Machine Learning.* MIT Press.
- Shahriari, B. et al. (2016). *Taking the Human Out of the Loop.* Proceedings of the IEEE.
- Zarchan, P. (2012). *Tactical and Strategic Missile Guidance.* AIAA.

```
구현 환경: Python 3.9 | numpy 1.24 | scipy 1.10 | matplotlib 3.7
외부 BO 라이브러리 미사용 (scikit-optimize, GPyOpt, botorch 등 제외)
```